# LeetCode 2200 — Find All K-Distant Indices in an Array bằng Python

Link đề: https://leetcode.com/problems/find-all-k-distant-indices-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def findKDistantIndices(self, nums: List[int], key: int, k: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **k-distant index**: chỉ số `i` mà trong vòng khoảng cách `k` quanh nó có ít nhất một phần tử bằng `key`.
2. Dùng được `abs`, `range`, `max/min` để **kẹp** khoảng `[j-k, j+k]` vào biên `[0, n-1]`.
3. Viết 4 cách: brute force `O(n²)`, đánh dấu bằng `set`, quét khoảng tuyến tính `O(n)` và mảng hiệu (difference array).
4. Hiểu tại sao kết quả tự nhiên đã **tăng dần** nếu ta duyệt theo thứ tự, và khi nào phải `sorted`.
5. Tránh lỗi hay gặp: thêm trùng chỉ số, quên kẹp biên, quên điều kiện `abs`, sai thứ tự kết quả.

## 2. Đề bài & ví dụ

Cho mảng `nums` (đánh số từ 0) và hai số nguyên `key`, `k`.

Một chỉ số `i` được gọi là **k-distant index** nếu tồn tại chỉ số `j` sao cho:

- `|i - j| <= k`, và
- `nums[j] == key`.

Hãy trả về **danh sách mọi k-distant index, sắp xếp tăng dần**.

| Ví dụ | nums | key | k | Output |
|---|---|---|---|---|
| 1 | `[3,4,9,1,3,9,5]` | `9` | `1` | `[1,2,3,4,5,6]` |
| 2 | `[2,2,2,2,2]` | `2` | `2` | `[0,1,2,3,4]` |

Ví dụ 1: `9` nằm ở chỉ số `2` và `5`. Quanh `2` (±1) ta có `1,2,3`; quanh `5` ta có `4,5,6`. Hợp lại là `[1,2,3,4,5,6]`.

Constraints:

- `1 <= nums.length <= 1000`
- `1 <= nums[i] <= 1000`
- `key` là một số nguyên có trong `nums`.
- `1 <= k <= nums.length`

In [1]:
examples = [([3, 4, 9, 1, 3, 9, 5], 9, 1, [1, 2, 3, 4, 5, 6]), ([2, 2, 2, 2, 2], 2, 2, [0, 1, 2, 3, 4])]
for nums, key, k, expected in examples:
    key_idx = [j for j, v in enumerate(nums) if v == key]
    print("nums =", nums, "| key =", key, "| k =", k, "| chỉ số của key:", key_idx, "| mong đợi:", expected)

nums = [3, 4, 9, 1, 3, 9, 5] | key = 9 | k = 1 | chỉ số của key: [2, 5] | mong đợi: [1, 2, 3, 4, 5, 6]
nums = [2, 2, 2, 2, 2] | key = 2 | k = 2 | chỉ số của key: [0, 1, 2, 3, 4] | mong đợi: [0, 1, 2, 3, 4]


## 3. Kiến thức nền

### 3.1. `enumerate` — lấy cả chỉ số lẫn giá trị

`enumerate(nums)` sinh các cặp `(chỉ_số, giá_trị)`. Dùng khi vừa cần biết vị trí `j`, vừa cần so sánh `nums[j]`.

In [2]:
nums = [3, 4, 9, 1, 3, 9, 5]
for j, v in enumerate(nums):
    print(j, v)
print([j for j, v in enumerate(nums) if v == 9])

0 3
1 4
2 9
3 1
4 3
5 9
6 5
[2, 5]


### 3.2. Khoảng cách `abs(i - j)`

`abs(x)` là giá trị tuyệt đối. `|i - j| <= k` nghĩa là `i` nằm trong đoạn `[j - k, j + k]` (hai phía quanh `j`).

In [3]:
i, j, k = 4, 2, 2
print(abs(i - j), abs(i - j) <= k)
print(abs(1 - 5), abs(1 - 5) <= 2)
print(abs(-7), abs(7))

2 True
4 False
7 7


### 3.3. `max`, `min` để kẹp biên

Đoạn `[j - k, j + k]` có thể tràn ra ngoài mảng (âm hoặc `>= n`). Ta kẹp lại: `lo = max(0, j - k)`, `hi = min(n - 1, j + k)`.

In [4]:
n = 7
for j, k in [(0, 2), (3, 2), (6, 3)]:
    lo, hi = max(0, j - k), min(n - 1, j + k)
    print(f"j={j}, k={k} -> [{j-k}, {j+k}] kẹp thành [{lo}, {hi}]")

j=0, k=2 -> [-2, 2] kẹp thành [0, 2]
j=3, k=2 -> [1, 5] kẹp thành [1, 5]
j=6, k=3 -> [3, 9] kẹp thành [3, 6]


### 3.4. `range(lo, hi + 1)` và `set`

`range(a, b)` **không** gồm `b`, nên muốn gồm `hi` phải viết `range(lo, hi + 1)`. `set` là tập hợp không trùng lặp, thêm bằng `s.add(x)` hoặc `s.update(iterable)`; `sorted(s)` trả về danh sách tăng dần.

In [5]:
s = set()
s.update(range(1, 4))     # {1,2,3}
s.update(range(3, 6))     # {3,4,5} -> 3 trùng, bị bỏ
print(s, sorted(s))
print(list(range(2, 2 + 1)), list(range(5, 4)))   # range rỗng khi a > b-1

{1, 2, 3, 4, 5} [1, 2, 3, 4, 5]
[2] []


## 4. Ý tưởng nền tảng — tính bằng tay

Lấy `nums = [3,4,9,1,3,9,5]`, `key = 9`, `k = 1`, `n = 7`.

| Bước | Việc làm | Kết quả |
|---|---|---|
| 1 | Tìm chỉ số của `key` | `j = 2` và `j = 5` |
| 2 | Quanh `j = 2` | `[1, 3]` → chỉ số `1, 2, 3` |
| 3 | Quanh `j = 5` | `[4, 6]` → chỉ số `4, 5, 6` |
| 4 | Gộp, bỏ trùng, sắp xếp | `[1,2,3,4,5,6]` |

Hai cách nhìn tương đương:

- **Nhìn từ `i`:** "xung quanh `i` có `key` không?"
- **Nhìn từ `j`:** "mỗi `key` ở `j` làm sáng các chỉ số `j-k ... j+k`."

Cách nhìn thứ hai cho thuật toán nhanh hơn vì ta chỉ cần xử lý các `j` có `key`.

In [6]:
nums, key, k = [3, 4, 9, 1, 3, 9, 5], 9, 1
n = len(nums)
lit = set()
for j in range(n):
    if nums[j] == key:
        lo, hi = max(0, j - k), min(n - 1, j + k)
        print(f"key ở j={j}: làm sáng [{lo}, {hi}]")
        lit.update(range(lo, hi + 1))
print(sorted(lit))

key ở j=2: làm sáng [1, 3]
key ở j=5: làm sáng [4, 6]
[1, 2, 3, 4, 5, 6]


## 5. Cách 1 — Brute Force: với mỗi `i`, tìm `j`

Đúng theo định nghĩa: với mỗi chỉ số `i`, duyệt mọi `j` và kiểm tra `abs(i - j) <= k and nums[j] == key`. Nếu có một `j` thỏa thì thêm `i` và `break`. Vì `i` duyệt tăng dần nên kết quả tự động tăng dần.

In [7]:
class SolutionBrute:
    def findKDistantIndices(self, nums, key, k):
        n = len(nums)
        res = []
        for i in range(n):
            for j in range(n):
                if abs(i - j) <= k and nums[j] == key:
                    res.append(i)
                    break            # đã tìm được một j, khỏi xét tiếp
        return res

sol = SolutionBrute()
print(sol.findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1))
print(sol.findKDistantIndices([2, 2, 2, 2, 2], 2, 2))

[1, 2, 3, 4, 5, 6]
[0, 1, 2, 3, 4]


**Độ phức tạp:** Time `O(n²)` (hai vòng lặp lồng), Space `O(1)` ngoài kết quả.

**Nút thắt:** với mỗi `i` ta quét lại cả mảng để tìm `key`, trong khi vị trí của `key` là cố định. Ta lặp đi lặp lại cùng một công việc "tìm key".

## 6. Ý tưởng thuật toán — bước đột phá

**Đảo góc nhìn:** thay vì hỏi "với mỗi `i`, có `key` gần không?", ta hỏi "mỗi `key` ở `j` làm sáng những `i` nào?" — câu trả lời là đoạn `[j-k, j+k]` (đã kẹp biên).

Kết quả là **hợp của các đoạn** quanh mỗi `key`. Quy tắc:

- **Nếu** `nums[j] == key` **thì** đánh dấu mọi chỉ số trong `[max(0,j-k), min(n-1,j+k)]`.
- Hợp các đoạn đó, bỏ trùng, sắp tăng dần chính là đáp án.

**Invariant (cho cách quét tuyến tính):** các `j` được xét tăng dần ⇒ đoạn bắt đầu của chúng không giảm ⇒ ta chỉ cần nhớ `nxt` = chỉ số nhỏ nhất **chưa** được thêm; mỗi lần chỉ thêm phần mới `[max(nxt, j-k), hi]`, nhờ đó không trùng và vẫn tăng dần.

## 7. Các cách tiếp theo

### Cách 2 — Đánh dấu bằng `set` rồi `sorted`

Với mỗi `key` ở `j`, thêm cả đoạn `range(lo, hi+1)` vào `set`. Cuối cùng trả `sorted(set)`.

In [8]:
class SolutionSet:
    def findKDistantIndices(self, nums, key, k):
        n = len(nums)
        lit = set()
        for j in range(n):
            if nums[j] == key:
                lit.update(range(max(0, j - k), min(n - 1, j + k) + 1))
        return sorted(lit)

sol = SolutionSet()
print(sol.findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1))
print(sol.findKDistantIndices([2, 2, 2, 2, 2], 2, 2))

[1, 2, 3, 4, 5, 6]
[0, 1, 2, 3, 4]


**Độ phức tạp:** mỗi `key` thêm tối đa `2k + 1` phần tử ⇒ Time `O(n·k)` ở tệ nhất (cộng `O(n log n)` sắp xếp), Space `O(n)`. Với `n <= 1000` rất ổn nhưng chưa tối ưu.

### Cách 3 — Quét khoảng tuyến tính (không trùng, tự sắp)

Duy trì `nxt` (chỉ số nhỏ nhất chưa thêm). Với mỗi `key` ở `j` (theo thứ tự tăng), thêm các chỉ số từ `max(nxt, j-k)` đến `min(n-1, j+k)`, rồi cập nhật `nxt = hi + 1`. Mỗi chỉ số được thêm **đúng một lần**.

In [9]:
class SolutionSweep:
    def findKDistantIndices(self, nums, key, k):
        n = len(nums)
        res = []
        nxt = 0                                   # chỉ số nhỏ nhất chưa được thêm
        for j in range(n):
            if nums[j] == key:
                lo = max(nxt, j - k)              # bỏ phần đã thêm
                hi = min(n - 1, j + k)
                res.extend(range(lo, hi + 1))     # rỗng nếu lo > hi
                nxt = max(nxt, hi + 1)
        return res

sol = SolutionSweep()
print(sol.findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1))
print(sol.findKDistantIndices([2, 2, 2, 2, 2], 2, 2))

[1, 2, 3, 4, 5, 6]
[0, 1, 2, 3, 4]


**Độ phức tạp:** Time `O(n)` (mỗi chỉ số được duyệt `j` một lần và được thêm vào kết quả tối đa một lần), Space `O(1)` ngoài kết quả.

### Cách 4 — Mảng hiệu (difference array) + tổng tiền tố

Kỹ thuật "cộng 1 lên một đoạn": `diff[lo] += 1; diff[hi + 1] -= 1`. Sau đó tính tổng chạy `run`; chỉ số `i` có `run > 0` nghĩa là nằm trong ít nhất một đoạn. Kỹ thuật này tổng quát khi đề có **rất nhiều** đoạn.

In [10]:
class SolutionDiff:
    def findKDistantIndices(self, nums, key, k):
        n = len(nums)
        diff = [0] * (n + 1)
        for j in range(n):
            if nums[j] == key:
                diff[max(0, j - k)] += 1
                diff[min(n - 1, j + k) + 1] -= 1
        res, run = [], 0
        for i in range(n):
            run += diff[i]
            if run > 0:
                res.append(i)
        return res

sol = SolutionDiff()
print(sol.findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1))
print(sol.findKDistantIndices([2, 2, 2, 2, 2], 2, 2))

[1, 2, 3, 4, 5, 6]
[0, 1, 2, 3, 4]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `diff`. Nhược điểm: tốn thêm mảng; ưu điểm: dễ mở rộng (ví dụ đếm **bao nhiêu** `key` phủ mỗi chỉ số — chính là giá trị `run`).

## 8. TRACE từng bước

Trace cách 3 (quét khoảng) với `nums = [3,4,9,1,3,9,5]`, `key = 9`, `k = 1`:

| j | nums[j] | là key? | lo | hi | thêm | nxt | res |
|---|---|---|---|---|---|---|---|
| 2 | 9 | có | max(0,1)=1 | min(6,3)=3 | 1,2,3 | 4 | [1,2,3] |
| 5 | 9 | có | max(4,4)=4 | min(6,6)=6 | 4,5,6 | 7 | [1,2,3,4,5,6] |

Các `j` khác không phải `key` nên bị bỏ qua. Hàm dưới in từng bước.

In [11]:
def trace_sweep(nums, key, k):
    n = len(nums)
    res, nxt = [], 0
    for j in range(n):
        if nums[j] == key:
            lo, hi = max(nxt, j - k), min(n - 1, j + k)
            added = list(range(lo, hi + 1))
            res.extend(added)
            nxt = max(nxt, hi + 1)
            print(f"j={j} | lo={lo} hi={hi} | thêm {added} | nxt={nxt} | res={res}")
        else:
            print(f"j={j} | nums[j]={nums[j]} không phải key")
    return res

print(trace_sweep([3, 4, 9, 1, 3, 9, 5], 9, 1))

j=0 | nums[j]=3 không phải key
j=1 | nums[j]=4 không phải key
j=2 | lo=1 hi=3 | thêm [1, 2, 3] | nxt=4 | res=[1, 2, 3]
j=3 | nums[j]=1 không phải key
j=4 | nums[j]=3 không phải key
j=5 | lo=4 hi=6 | thêm [4, 5, 6] | nxt=7 | res=[1, 2, 3, 4, 5, 6]
j=6 | nums[j]=5 không phải key
[1, 2, 3, 4, 5, 6]


In [12]:
# trace với các đoạn chồng lên nhau: key ở 1 và 3, k = 2
print(trace_sweep([7, 5, 7, 5, 7, 7], 5, 2))

j=0 | nums[j]=7 không phải key
j=1 | lo=0 hi=3 | thêm [0, 1, 2, 3] | nxt=4 | res=[0, 1, 2, 3]
j=2 | nums[j]=7 không phải key
j=3 | lo=4 hi=5 | thêm [4, 5] | nxt=6 | res=[0, 1, 2, 3, 4, 5]
j=4 | nums[j]=7 không phải key
j=5 | nums[j]=7 không phải key
[0, 1, 2, 3, 4, 5]


## 9. Code nộp LeetCode

Dùng cách 3 (quét khoảng): `O(n)`, không cần `set`, kết quả đã tăng dần.

In [13]:
from typing import List

class Solution:
    def findKDistantIndices(self, nums: List[int], key: int, k: int) -> List[int]:
        n = len(nums)
        res = []
        nxt = 0
        for j in range(n):
            if nums[j] == key:
                lo = max(nxt, j - k)
                hi = min(n - 1, j + k)
                res.extend(range(lo, hi + 1))
                nxt = max(nxt, hi + 1)
        return res

print(Solution().findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1))
print(Solution().findKDistantIndices([2, 2, 2, 2, 2], 2, 2))

[1, 2, 3, 4, 5, 6]
[0, 1, 2, 3, 4]


## 10. Vì sao thuật toán đúng?

- **Đúng tập hợp:** `i` là k-distant ⇔ tồn tại `j` có `nums[j] == key` và `j - k <= i <= j + k` (kèm `0 <= i <= n-1`) ⇔ `i` thuộc một đoạn kẹp biên `[max(0,j-k), min(n-1,j+k)]` của một `key`. Ta lấy đúng hợp các đoạn đó.
- **Không trùng:** khi xử lý `key` ở `j`, mọi chỉ số `< nxt` đã được thêm rồi, nên ta bắt đầu từ `max(nxt, j-k)`. Đoạn mới nằm hoàn toàn bên phải phần đã thêm.
- **Tăng dần:** `j` tăng ⇒ `hi = min(n-1, j+k)` không giảm ⇒ `nxt` không giảm; các đoạn mới luôn bắt đầu từ `>= nxt`, nên danh sách luôn được nối thêm phần lớn hơn.
- **Không sót:** phần bị bỏ `[j-k, nxt-1]` đã được thêm bởi `key` trước đó (đoạn của nó phủ tới `nxt - 1`), nên không mất chỉ số nào.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `m` là số lần `key` xuất hiện (`m <= n`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. `set` + `sorted` | `O(m·k + n log n)` ≤ `O(n²)` | `O(n)` |
| 3. Quét khoảng | `O(n)` | `O(1)` (ngoài kết quả) |
| 4. Mảng hiệu | `O(n)` | `O(n)` |

Kết quả có thể dài tới `n` phần tử, nên Space "thực" của mọi cách tối thiểu `O(n)` nếu tính cả output.

## 12. Bộ test

Test ví dụ đề, edge case, rồi đối chiếu cả 5 lớp trên input ngẫu nhiên (có `seed`).

In [14]:
solutions = [SolutionBrute(), SolutionSet(), SolutionSweep(), SolutionDiff(), Solution()]

for s in solutions:
    assert s.findKDistantIndices([3, 4, 9, 1, 3, 9, 5], 9, 1) == [1, 2, 3, 4, 5, 6]
    assert s.findKDistantIndices([2, 2, 2, 2, 2], 2, 2) == [0, 1, 2, 3, 4]
print("ví dụ đề: OK")

ví dụ đề: OK


In [15]:
for s in solutions:
    assert s.findKDistantIndices([5], 5, 1) == [0]                         # một phần tử
    assert s.findKDistantIndices([1, 2, 3, 4, 5], 5, 1) == [3, 4]          # key ở cuối: kẹp biên phải
    assert s.findKDistantIndices([5, 2, 3, 4, 1], 5, 2) == [0, 1, 2]       # key ở đầu: kẹp biên trái
    assert s.findKDistantIndices([1, 9, 2, 3, 4, 5, 9, 6], 9, 1) == [0, 1, 2, 5, 6, 7]  # hai đoạn rời nhau
    assert s.findKDistantIndices([9, 1, 9], 9, 3) == [0, 1, 2]             # k >= n, hai đoạn chồng
print("edge case: OK")

edge case: OK


In [16]:
import random
random.seed(2200)

def reference(nums, key, k):
    """Định nghĩa thẳng: i hợp lệ nếu tồn tại j."""
    n = len(nums)
    return [i for i in range(n) if any(abs(i - j) <= k and nums[j] == key for j in range(n))]

count = 0
for _ in range(800):
    n = random.randint(1, 25)
    nums = [random.randint(1, 5) for _ in range(n)]
    key = random.choice(nums)
    k = random.randint(1, n)
    expected = reference(nums, key, k)
    for s in solutions:
        got = s.findKDistantIndices(nums, key, k)
        assert got == expected, (type(s).__name__, nums, key, k, got, expected)
    count += 1
print("random đối chiếu:", count, "ca x", len(solutions), "cách: OK")

random đối chiếu: 800 ca x 5 cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thêm trùng chỉ số khi các đoạn chồng nhau

Nếu cứ `res.extend(range(...))` cho mỗi `key` mà không bỏ phần đã thêm, các đoạn chồng nhau cho ra chỉ số lặp.

In [17]:
def wrong_duplicates(nums, key, k):
    n, res = len(nums), []
    for j in range(n):
        if nums[j] == key:
            res.extend(range(max(0, j - k), min(n - 1, j + k) + 1))   # không chống trùng
    return res

nums, key, k = [2, 2, 2], 2, 1
print("sai :", wrong_duplicates(nums, key, k))
print("đúng:", Solution().findKDistantIndices(nums, key, k))

sai : [0, 1, 0, 1, 2, 1, 2]
đúng: [0, 1, 2]


### Lỗi 2 — Quên kẹp biên

`range(j - k, j + k + 1)` có thể chứa số âm hoặc `>= n`. Số âm còn "hợp lệ" với Python (chỉ số âm) nên lỗi rất khó phát hiện khi dùng để đọc mảng.

In [18]:
def wrong_no_clamp(nums, key, k):
    n, lit = len(nums), set()
    for j in range(n):
        if nums[j] == key:
            lit.update(range(j - k, j + k + 1))      # SAI: không kẹp
    return sorted(lit)

print("sai :", wrong_no_clamp([9, 1, 2], 9, 2))       # có chỉ số âm / vượt biên
print("đúng:", Solution().findKDistantIndices([9, 1, 2], 9, 2))

sai : [-2, -1, 0, 1, 2]
đúng: [0, 1, 2]


### Lỗi 3 — Dùng `i - j <= k` thay vì `abs(i - j) <= k`

Bỏ `abs` thì chỉ xét phía bên phải `j`, còn `i < j` (bên trái) luôn cho `i - j` âm nên "đúng" vô điều kiện, dẫn tới kết quả thừa.

In [19]:
def wrong_no_abs(nums, key, k):
    n = len(nums)
    return [i for i in range(n) if any((i - j) <= k and nums[j] == key for j in range(n))]

print("sai :", wrong_no_abs([1, 1, 1, 9], 9, 1))    # mọi i < 3 đều bị tính
print("đúng:", Solution().findKDistantIndices([1, 1, 1, 9], 9, 1))

sai : [0, 1, 2, 3]
đúng: [2, 3]


### Lỗi 4 — Dùng `range(lo, hi)` thay vì `range(lo, hi + 1)`

`range` bỏ phần tử cuối. Quên `+ 1` làm mất chỉ số `j + k`.

In [20]:
n, j, k = 7, 2, 1
print("sai :", list(range(max(0, j - k), min(n - 1, j + k))))        # thiếu 3
print("đúng:", list(range(max(0, j - k), min(n - 1, j + k) + 1)))

sai : [1, 2]
đúng: [1, 2, 3]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Trong vòng `k` quanh phần tử đặc biệt**" ⇒ mỗi phần tử đặc biệt phủ một đoạn `[j-k, j+k]`; bài là **hợp các đoạn**.
- Hai kỹ thuật hợp đoạn: **quét với `nxt`** (đoạn đã sắp tăng theo `j`) và **mảng hiệu** (đoạn bất kỳ).
- Luôn kẹp biên bằng `max(0, ·)` và `min(n-1, ·)`.
- Khi `n <= 1000` brute force vẫn qua, nhưng hãy luyện cách `O(n)`.
- Họ hàng: Merge Intervals, Meeting Rooms, Range Addition, Corporate Flight Bookings.

## 15. Biến thể của bài

1. **Đếm số k-distant index:** chỉ cần `len(...)` hoặc đếm khi quét.
2. **Chỉ nhìn phía bên phải:** `0 <= j - i <= k` (đoạn trở thành `[j-k, j]`).
3. **Khoảng cách theo giá trị:** thay vì chỉ số, xét `|nums[i] - nums[j]| <= k`.
4. **Nhiều `key`:** `nums[j]` thuộc một tập `keys` cho trước.
5. **Đếm độ phủ:** với mỗi `i`, có bao nhiêu `key` trong vòng `k` — dùng mảng hiệu (giá trị `run`).

In [21]:
# Biến thể 5: độ phủ của từng chỉ số bằng mảng hiệu
def coverage(nums, key, k):
    n = len(nums)
    diff = [0] * (n + 1)
    for j in range(n):
        if nums[j] == key:
            diff[max(0, j - k)] += 1
            diff[min(n - 1, j + k) + 1] -= 1
    out, run = [], 0
    for i in range(n):
        run += diff[i]
        out.append(run)
    return out

print(coverage([3, 4, 9, 1, 3, 9, 5], 9, 1))   # mỗi chỉ số 1..6 được phủ đúng một lần

[0, 1, 1, 1, 1, 1, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Với mỗi `i`, quét `j` | `O(n²)` | `O(1)` | Hiểu định nghĩa, kiểm tra đáp án |
| 2. `set` + `sorted` | Mỗi `key` thêm cả đoạn vào set | `O(m·k + n log n)` | `O(n)` | Viết nhanh, `n` nhỏ |
| 3. Quét khoảng | Nhớ `nxt`, chỉ thêm phần mới | `O(n)` | `O(1)` | **Nộp bài**, tối ưu và gọn |
| 4. Mảng hiệu | Cộng đoạn bằng `diff`, tổng tiền tố | `O(n)` | `O(n)` | Nhiều đoạn bất kỳ, cần độ phủ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `findKDistantIndices([1,2,3,2,1], 3, 1)` trả về gì?

**Bài 2 (tính tay).** Với `nums = [8,1,1,8,1,1,1,1,8]`, `key = 8`, `k = 2`, hãy liệt kê đoạn phủ của từng `key` rồi suy ra đáp án.

**Bài 3 (điền TODO).** Hoàn thành `clamp_range(j, k, n)` trả về cặp `(lo, hi)` đã kẹp biên.

**Bài 4 (tự viết lại).** Viết hàm `count_k_distant(nums, key, k)` đếm số k-distant index mà không tạo danh sách kết quả.

**Bài 5 (mini challenge).** Viết `findKDistantValues(nums, key, k)` trả về **giá trị** `nums[i]` (không phải chỉ số) tại các k-distant index, theo thứ tự chỉ số tăng dần.

In [22]:
# Bài 1: chạy để kiểm tra dự đoán của bạn
print(Solution().findKDistantIndices([1, 2, 3, 2, 1], 3, 1))

[1, 2, 3]


In [23]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def clamp_range(j, k, n):
    # TODO: trả về (lo, hi) với lo = max(0, j-k), hi = min(n-1, j+k)
    pass

def count_k_distant(nums, key, k):
    # TODO: đếm số k-distant index, tối ưu O(n)
    pass

def find_k_distant_values(nums, key, k):
    # TODO: trả về giá trị nums[i] tại các k-distant index
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: Chỉ số trái nhất có thể là `j - k` nhưng không được âm.
- Mức 2: Chỉ số phải nhất là `j + k` nhưng không được vượt `n - 1`.
- Mức 3: `return max(0, j - k), min(n - 1, j + k)`.

**Bài 4**
- Mức 1: Dùng lại cách quét khoảng.
- Mức 2: Thay vì `extend`, cộng độ dài đoạn mới `hi - lo + 1` (nếu `lo <= hi`).
- Mức 3: `total += max(0, hi - lo + 1)`, cập nhật `nxt` như cũ.

**Bài 5**
- Mức 1: Lấy danh sách chỉ số từ lời giải chính.
- Mức 2: Đổi mỗi chỉ số thành `nums[i]`.
- Mức 3: `[nums[i] for i in Solution().findKDistantIndices(nums, key, k)]`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [24]:
# Bài 1: key=3 ở chỉ số 2, k=1 -> [1,2,3]
assert Solution().findKDistantIndices([1, 2, 3, 2, 1], 3, 1) == [1, 2, 3]

# Bài 2: key ở 0, 3, 8; k=2 -> [0,2], [1,5], [6,8] -> hợp = 0..8 trừ không sót
ans2 = Solution().findKDistantIndices([8, 1, 1, 8, 1, 1, 1, 1, 8], 8, 2)
assert ans2 == [0, 1, 2, 3, 4, 5, 6, 7, 8], ans2

# Bài 3
def clamp_range_ans(j, k, n):
    return max(0, j - k), min(n - 1, j + k)
assert clamp_range_ans(0, 2, 7) == (0, 2)
assert clamp_range_ans(6, 3, 7) == (3, 6)

# Bài 4
def count_k_distant_ans(nums, key, k):
    n, total, nxt = len(nums), 0, 0
    for j in range(n):
        if nums[j] == key:
            lo, hi = max(nxt, j - k), min(n - 1, j + k)
            total += max(0, hi - lo + 1)
            nxt = max(nxt, hi + 1)
    return total
assert count_k_distant_ans([3, 4, 9, 1, 3, 9, 5], 9, 1) == 6
assert count_k_distant_ans([2, 2, 2, 2, 2], 2, 2) == 5

# Bài 5
def find_k_distant_values_ans(nums, key, k):
    return [nums[i] for i in Solution().findKDistantIndices(nums, key, k)]
assert find_k_distant_values_ans([3, 4, 9, 1, 3, 9, 5], 9, 1) == [4, 9, 1, 3, 9, 5]
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from typing import List

class Solution:
    def findKDistantIndices(self, nums: List[int], key: int, k: int) -> List[int]:
        n = len(nums)
        res, nxt = [], 0                     # nxt: chỉ số nhỏ nhất chưa thêm
        for j in range(n):
            if nums[j] == key:
                lo, hi = max(nxt, j - k), min(n - 1, j + k)
                res.extend(range(lo, hi + 1))
                nxt = max(nxt, hi + 1)
        return res
```

### Checklist

- [ ] Hiểu `|i - j| <= k` là đoạn `[j-k, j+k]` quanh mỗi `key`.
- [ ] Kẹp biên: `max(0, ·)` và `min(n-1, ·)`.
- [ ] `range(lo, hi + 1)` — nhớ `+ 1`.
- [ ] Không thêm trùng (dùng `nxt` hoặc `set`) và kết quả tăng dần.
- [ ] Đã test `key` ở đầu/cuối mảng, `k >= n`, một phần tử.
- [ ] Biết cách mảng hiệu cho bài nhiều đoạn.